In [ ]:
# Developing a recommendation system using collaborative filtering or deep learning approaches.
import torch
import torch.nn as nn
import torch.optim as optim
import random
# ------------------------
# Original positive dataset: (user_id, item_id, rating)
# Movies: 0 = Inception, 1 = Avatar, 2 = Titanic, 3 = Matrix, 4 = Interstellar
# ------------------------
positive_data = [
    (0, 0, 1), (0, 1, 1), (0, 3, 1),       # User 0 likes Inception, Avatar, Matrix
    (1, 1, 1), (1, 2, 1),                  # User 1 likes Avatar, Titanic
    (2, 0, 1), (2, 2, 1), (2, 4, 1),       # User 2 likes Inception, Titanic, Interstellar
    (3, 3, 1), (3, 4, 1)                   # User 3 likes Matrix, Interstellar
]

num_users = 4
num_items = 5
embed_dim = 8
neg_per_pos = 2  # negatives per positive

# ------------------------
# Generate negative samples
# ------------------------
all_items = list(range(num_items))
positive_dict = {}
for u, i, r in positive_data:
    positive_dict.setdefault(u, set()).add(i)

data = positive_data.copy()
for u, i, r in positive_data:
    neg_count = 0
    while neg_count < neg_per_pos:
        neg_item = random.choice(all_items)
        if neg_item not in positive_dict[u]:
            data.append((u, neg_item, 0))  # mark as negative
            neg_count += 1

print(f"Training samples: {len(data)}")

# ------------------------
# Collaborative Deep Learning Model
# ------------------------
class CFModel(nn.Module):
    def __init__(self, n_users, n_items, emb_dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.fc = nn.Sequential(
            nn.Linear(emb_dim * 2, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, u, i):
        u_emb = self.user_emb(u)
        i_emb = self.item_emb(i)
        x = torch.cat([u_emb, i_emb], dim=1)
        return self.fc(x)

# ------------------------
# Prepare data
# ------------------------
users = torch.tensor([u for u, i, r in data])
items = torch.tensor([i for u, i, r in data])
ratings = torch.tensor([r for u, i, r in data], dtype=torch.float32)

# ------------------------
# Train model
# ------------------------
model = CFModel(num_users, num_items, embed_dim)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(50):
    optimizer.zero_grad()
    preds = model(users, items).squeeze()
    loss = criterion(preds, ratings)
    loss.backward()
    optimizer.step()

# ------------------------
# Make predictions for a given user
# ------------------------
user_id = 0
test_items = torch.arange(num_items)
scores = model(torch.tensor([user_id]*num_items), test_items).detach().numpy().flatten()

movies = ["Inception", "Avatar", "Titanic", "Matrix", "Interstellar"]
print(f"\nRecommendations for user {user_id}:")
for m, s in sorted(zip(movies, scores), key=lambda x: x[1], reverse=True):
    print(f"{m}: {float(s):.3f}")


Training samples: 30

Recommendations for user 0:
Avatar: 0.987
Inception: 0.969
Matrix: 0.962
Interstellar: 0.024
Titanic: 0.007
